In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

In [3]:
CSV_PATH = "mitbih_train.csv"
OUTPUT_DIR = "ecg_images"
SAMPLES_PER_CLASS = 500

In [4]:
class_map = {
    0: "Normal",
    1: "Supraventricular",
    2: "Ventricular",
    3: "Fusion",
    4: "Others"
}

In [5]:
for cls_name in class_map.values():
    os.makedirs(os.path.join(OUTPUT_DIR, cls_name), exist_ok=True)

In [6]:
df = pd.read_csv(CSV_PATH, header=None)

In [7]:
def augment_signal(signal):
    # Add slight Gaussian noise
    noise = np.random.normal(0, 0.01, len(signal))
    return signal + noise

In [8]:
def save_ecg_image(signal, save_path):
    plt.figure(figsize=(3,3))
    plt.plot(signal, color='black')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
    plt.close()

In [9]:
for label in range(5):
    print(f"\nProcessing Class {label} - {class_map[label]}")

    class_df = df[df.iloc[:, -1] == label]

    save_count = 0
    cls_name = class_map[label]
    save_folder = os.path.join(OUTPUT_DIR, cls_name)

    # Enough samples → Random Sampling
    if len(class_df) >= SAMPLES_PER_CLASS:
        sampled_df = class_df.sample(SAMPLES_PER_CLASS, random_state=42)

        for i, row in tqdm(sampled_df.iterrows(), total=SAMPLES_PER_CLASS):
            signal = row[:-1].values
            save_path = os.path.join(save_folder, f"{save_count}.png")
            save_ecg_image(signal, save_path)
            save_count += 1

    # Not enough → Augmentation
    else:
        print(f"Augmenting Class {label} (only {len(class_df)} samples)")

        # Save original samples
        for i, row in tqdm(class_df.iterrows(), total=len(class_df)):
            signal = row[:-1].values
            save_path = os.path.join(save_folder, f"{save_count}.png")
            save_ecg_image(signal, save_path)
            save_count += 1

        # Generate augmented samples
        while save_count < SAMPLES_PER_CLASS:
            row = class_df.sample(1).iloc[0]
            signal = row[:-1].values

            aug_signal = augment_signal(signal)

            save_path = os.path.join(save_folder, f"{save_count}.png")
            save_ecg_image(aug_signal, save_path)

            save_count += 1

    print(f"Saved {save_count} images for {cls_name}")

print("\n DONE: Dataset Ready!")


Processing Class 0 - Normal


100%|█████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [00:10<00:00, 45.68it/s]


Saved 500 images for Normal

Processing Class 1 - Supraventricular


100%|█████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [00:10<00:00, 46.74it/s]


Saved 500 images for Supraventricular

Processing Class 2 - Ventricular


...

Saved 500 images for Ventricular

Processing Class 3 - Fusion


100%|█████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [00:11<00:00, 44.78it/s]


Saved 500 images for Fusion

Processing Class 4 - Others


100%|█████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [00:11<00:00, 42.70it/s]

Saved 500 images for Others

 DONE: Dataset Ready!
